In [1]:
## MES (제조 실행 시스템) :생산 실적, 작업지시 데이터 -> 생산량, 작업시간, 생산일자
## QMS (품질 관리 시스템) :검사결과, 불량 이력 -> 검사값, 합력/불합력, 불량 유형
## EAM (설비 자산 관리) : 설비정보, 정비이력 -> 설비명, 제조사, 정비비용
## IoT 플랫폼 : 센서 데이터 -> 온도, 압력, 진동

In [2]:
# 설비별 생산량은 많은데, 품질 불량률도 높은 설비는 어디냐? 생산 + 품질
# 정비 비용이 많이 드는 설비가 실제 생산 기여도가 낮다면? 생산+ 정비
# 특정 온도 범위에서 불량률이 증가하는가? 센서 + 품질
# 신규 설비와 구형 설비의 생산 효율 차이는? 설비정보 + 생산

In [3]:
# 1. concat 함수
# 2. marge 함수 (sql 의 join)
# 3. join (두 데이터프레임의 인덱스를 먼저 다 맞춰나야한다.)

In [4]:
import pandas as pd

In [5]:
production_df = pd.read_csv('./data/05_production.csv', parse_dates=['production_date']) #바로 date_time으로 읽어오는 법

In [6]:
production_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1872 entries, 0 to 1871
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   production_id    1872 non-null   int64         
 1   equipment_id     1872 non-null   object        
 2   product_code     1872 non-null   object        
 3   production_date  1872 non-null   datetime64[ns]
 4   start_time       1872 non-null   object        
 5   end_time         1872 non-null   object        
 6   target_quantity  1872 non-null   int64         
 7   actual_quantity  1872 non-null   int64         
 8   good_quantity    1872 non-null   int64         
 9   defect_quantity  1872 non-null   int64         
 10  cycle_time       1872 non-null   float64       
 11  work_order_no    1872 non-null   object        
 12  lot_no           1872 non-null   object        
 13  operator_id      1872 non-null   object        
 14  shift            1872 non-null   object 

In [7]:
# 1월의 생산 데이터와 2월의 생산 데이터를 따로 만들자.

In [8]:
production_df.head(2)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48


In [9]:
df_1=production_df[production_df['production_date'].dt.month ==1]

In [10]:
df_2=production_df[production_df['production_date'].dt.month ==2]

In [11]:
df_1

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
621,622,PRESS-002,BUMPER-A,2024-01-31,2024-01-31 21:36:00,2024-02-01 00:34:27,150,144,134,10,74.36,WO202401319043,LOT2024013100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
622,623,PRESS-002,BUMPER-A,2024-01-31,2024-02-01 01:00:00,2024-02-01 03:53:40,134,128,119,9,81.41,WO202401312435,LOT2024013100211,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
623,624,PRESS-002,DOOR-B,2024-01-31,2024-02-01 06:38:00,2024-02-01 09:13:32,141,135,125,10,69.13,WO202401314271,LOT2024013100212,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
624,625,ASM-001,BUMPER-A,2024-01-31,2024-01-31 10:18:00,2024-01-31 12:29:12,90,76,72,4,103.59,WO202401318758,LOT2024013100101,OP002,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48


In [12]:
df_2

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
626,627,INJ-001,DASH-C,2024-02-01,2024-02-01 08:23:00,2024-02-01 10:55:52,128,121,116,5,75.80,WO202402017052,LOT2024020100101,OP010,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
627,628,INJ-001,BUMPER-A,2024-02-01,2024-02-01 20:16:00,2024-02-01 22:47:50,136,129,120,9,70.62,WO202402018619,LOT2024020100110,OP005,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
628,629,INJ-002,BUMPER-A,2024-02-01,2024-02-01 10:03:00,2024-02-01 12:18:39,99,103,100,3,79.03,WO202402012584,LOT2024020100201,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
629,630,INJ-002,BUMPER-A,2024-02-01,2024-02-01 14:26:00,2024-02-01 17:06:01,109,114,111,3,84.22,WO202402011999,LOT2024020100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
630,631,INJ-002,DOOR-B,2024-02-01,2024-02-01 18:43:00,2024-02-01 21:09:28,140,147,144,3,59.79,WO202402014860,LOT2024020100203,OP010,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1223,1224,PRESS-002,BUMPER-A,2024-02-29,2024-02-29 22:29:00,2024-03-01 00:43:33,114,109,93,16,74.07,WO202402299728,LOT2024022900210,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1224,1225,PRESS-002,DASH-C,2024-02-29,2024-03-01 02:46:00,2024-03-01 04:37:35,87,83,69,14,80.67,WO202402296968,LOT2024022900211,OP005,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1225,1226,PRESS-002,DOOR-B,2024-02-29,2024-03-01 04:09:00,2024-03-01 06:05:17,118,113,96,17,61.75,WO202402297721,LOT2024022900212,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1226,1227,ASM-001,DASH-C,2024-02-29,2024-02-29 09:17:00,2024-02-29 12:14:22,116,98,84,14,108.59,WO202402295201,LOT2024022900101,OP002,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48


In [13]:
# column이 동일하면, 데이터를 합치기가 아주 쉽다.
pd.concat([df_1,df_2],ignore_index=True)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1223,1224,PRESS-002,BUMPER-A,2024-02-29,2024-02-29 22:29:00,2024-03-01 00:43:33,114,109,93,16,74.07,WO202402299728,LOT2024022900210,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1224,1225,PRESS-002,DASH-C,2024-02-29,2024-03-01 02:46:00,2024-03-01 04:37:35,87,83,69,14,80.67,WO202402296968,LOT2024022900211,OP005,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1225,1226,PRESS-002,DOOR-B,2024-02-29,2024-03-01 04:09:00,2024-03-01 06:05:17,118,113,96,17,61.75,WO202402297721,LOT2024022900212,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48
1226,1227,ASM-001,DASH-C,2024-02-29,2024-02-29 09:17:00,2024-02-29 12:14:22,116,98,84,14,108.59,WO202402295201,LOT2024022900101,OP002,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48


In [14]:
eq_df=pd.read_csv('./data/01_equipment.csv')

In [15]:
production_df.head(1)

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48


In [16]:
eq_df

,equipment_id,equipment_name,equipment_type,location,rated_capacity,installation_date,status,created_at,updated_at
0,INJ-001,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1,INJ-002,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
2,PRESS-001,프레스 1호기,프레스,A동 2라인,200.0,2019-05-10,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
3,PRESS-002,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
4,ASM-001,조립라인 1호기,조립라인,B동 1라인,100.0,2020-11-30,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00


In [17]:
pd.merge(production_df,eq_df,on='equipment_id')

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,created_at_x,updated_at_x,equipment_name,equipment_type,location,rated_capacity,installation_date,status,created_at_y,updated_at_y
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,...,2026-01-30 00:42:48,2026-01-30 00:42:48,조립라인 1호기,조립라인,B동 1라인,100.0,2020-11-30,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00


In [18]:
pd.merge(production_df,eq_df,on = 'equipment_id', how ='left') #how는 left의 데이터를 유실하지말고 right에는 없지만 left의 데이터에는 있는 것들은 NaN처리

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,created_at_x,updated_at_x,equipment_name,equipment_type,location,rated_capacity,installation_date,status,created_at_y,updated_at_y
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,...,2026-01-30 00:42:48,2026-01-30 00:42:48,조립라인 1호기,조립라인,B동 1라인,100.0,2020-11-30,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00


In [19]:
#생산데이터 + 설비정보데이터

In [20]:
step1=pd.merge(production_df,eq_df, on='equipment_id', how = 'left')

In [21]:
quality_df = pd.read_csv('./data/07_quality_inspection.csv')

In [22]:
quality_df.head(2)

,inspection_id,production_id,equipment_id,product_code,inspection_time,inspection_type,result,defect_code,measurement_value,measurement_unit,inspector_id,lot_no,sample_size,notes,created_at
0,1,1,INJ-001,BUMPER-A,2024-01-01 09:41:18,FINAL,PASS,\N,300.8279,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
1,2,1,INJ-001,BUMPER-A,2024-01-01 08:17:24,FINAL,PASS,\N,299.7696,mm,OP008,LOT2024010100101,1,NaN,2026-01-30 01:24:59


In [23]:
#생산데이터 + 설비데이터 + 품질데이터 -> 1개로 합치자

In [24]:
def get_count(x) :
    return (x=='FAIL').sum()

In [25]:
quality_sumary=quality_df.groupby('production_id').agg({'inspection_id':'count',
                                        'result': get_count,
                                        'measurement_value':'mean'})

In [26]:
quality_sumary

,inspection_id,result,measurement_value
production_id,,,
1,11,4,300.583600
2,13,6,299.203377
3,13,3,298.295115
4,11,2,400.704636
5,17,7,598.715941
...,...,...,...
1868,35,25,297.999489
1869,31,21,400.624397
1870,20,14,301.137370


In [27]:
step1

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,created_at_x,updated_at_x,equipment_name,equipment_type,location,rated_capacity,installation_date,status,created_at_y,updated_at_y
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,...,2026-01-30 00:42:48,2026-01-30 00:42:48,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,...,2026-01-30 00:42:48,2026-01-30 00:42:48,프레스 2호기,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,...,2026-01-30 00:42:48,2026-01-30 00:42:48,조립라인 1호기,조립라인,B동 1라인,100.0,2020-11-30,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00


In [28]:
comb=pd.merge(step1,quality_sumary,on='production_id')

In [29]:
comb.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at_x', 'updated_at_x',
       'equipment_name', 'equipment_type', 'location', 'rated_capacity',
       'installation_date', 'status', 'created_at_y', 'updated_at_y',
       'inspection_id', 'result', 'measurement_value'],
      dtype='object')

In [30]:
comb=comb.rename(columns={'inspection_id':'검사 건수','result':'불량건수','measurement_value' : '평균측정값'})

In [31]:
comb.to_csv('./data/comb.csv',index=False)

In [32]:
comb =pd.read_csv('./data/comb.csv')

In [33]:
comb

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,equipment_type,location,rated_capacity,installation_date,status,created_at_y,updated_at_y,검사 건수,불량건수,평균측정값
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,11,4,300.583600
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,13,6,299.203377
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,...,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,13,3,298.295115
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,...,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,11,2,400.704636
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,...,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,17,7,598.715941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,...,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,35,25,297.999489
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,...,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,31,21,400.624397
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,...,프레스,A동 2라인,200.0,2022-08-25,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,20,14,301.137370
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,...,조립라인,B동 1라인,100.0,2020-11-30,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00,30,20,300.775347


In [34]:
comb.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at_x', 'updated_at_x',
       'equipment_name', 'equipment_type', 'location', 'rated_capacity',
       'installation_date', 'status', 'created_at_y', 'updated_at_y', '검사 건수',
       '불량건수', '평균측정값'],
      dtype='object')

In [35]:
# 생산, 설비, 품질 검사

In [36]:
# 설비별로 생산 효율과, 품질 문제를 동시에 파악가능

In [37]:
#1. 어떤 설비가 생산량은 많은데, 불량률도 높은가?
#2. 설비 제조사별 품질 차이가 있는가?
#3. 특정 제품에서 불량이 집중되는지?

In [38]:
# 불량률계산 

In [39]:
comb['불량률']=(comb['불량건수']/comb['검사 건수'] *100).fillna(0)

In [40]:
#설비별 생산 및 품질 현환 (생산량 Top 5)

In [41]:
comb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1872 entries, 0 to 1871
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   production_id      1872 non-null   int64  
 1   equipment_id       1872 non-null   object 
 2   product_code       1872 non-null   object 
 3   production_date    1872 non-null   object 
 4   start_time         1872 non-null   object 
 5   end_time           1872 non-null   object 
 6   target_quantity    1872 non-null   int64  
 7   actual_quantity    1872 non-null   int64  
 8   good_quantity      1872 non-null   int64  
 9   defect_quantity    1872 non-null   int64  
 10  cycle_time         1872 non-null   float64
 11  work_order_no      1872 non-null   object 
 12  lot_no             1872 non-null   object 
 13  operator_id        1872 non-null   object 
 14  shift              1872 non-null   object 
 15  created_at_x       1872 non-null   object 
 16  updated_at_x       1872 

In [42]:
eq_df.head(2)

,equipment_id,equipment_name,equipment_type,location,rated_capacity,installation_date,status,created_at,updated_at
0,INJ-001,사출기 1호기,사출기,A동 1라인,150.0,2020-03-15,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00
1,INJ-002,사출기 2호기,사출기,A동 1라인,150.0,2021-06-20,ACTIVE,2024-01-01 00:00:00,2024-01-01 00:00:00


In [43]:
eq_summary = comb.groupby('equipment_name').agg({'production_id':'count',
                                  'actual_quantity':'sum',
                                   '검사 건수':'sum',
                                   '불량건수':'sum'})

In [44]:
eq_summary

,production_id,actual_quantity,검사 건수,불량건수
equipment_name,,,,
사출기 1호기,262,28163,5302,3017
사출기 2호기,430,51958,8528,4504
조립라인 1호기,234,22485,4581,2726
프레스 1호기,468,52069,9295,5123
프레스 2호기,478,51929,9711,5506


In [45]:
eq_summary.columns =['생산건수','총생산량','총검사수','불량건수']

In [46]:
eq_summary

,생산건수,총생산량,총검사수,불량건수
equipment_name,,,,
사출기 1호기,262,28163,5302,3017
사출기 2호기,430,51958,8528,4504
조립라인 1호기,234,22485,4581,2726
프레스 1호기,468,52069,9295,5123
프레스 2호기,478,51929,9711,5506


In [47]:
eq_summary['불량률']=(eq_summary['불량건수']/eq_summary['총검사수']*100).round(2)

In [48]:
eq_summary

,생산건수,총생산량,총검사수,불량건수,불량률
equipment_name,,,,,
사출기 1호기,262,28163,5302,3017,56.90
사출기 2호기,430,51958,8528,4504,52.81
조립라인 1호기,234,22485,4581,2726,59.51
프레스 1호기,468,52069,9295,5123,55.12
프레스 2호기,478,51929,9711,5506,56.70


In [49]:
eq_summary.sort_values(['총생산량','불량건수'], ascending= False)## 총생산량으로 정렬하고 같으면 불량건수 기준으로 정렬

,생산건수,총생산량,총검사수,불량건수,불량률
equipment_name,,,,,
프레스 1호기,468,52069,9295,5123,55.12
사출기 2호기,430,51958,8528,4504,52.81
프레스 2호기,478,51929,9711,5506,56.70
사출기 1호기,262,28163,5302,3017,56.90
조립라인 1호기,234,22485,4581,2726,59.51


In [50]:
## 정비 비용은 많이 들지만 생산 기여도가 낮은 설비는?

In [51]:
main_df= pd.read_csv('./data/10_maintenance_history.csv')

In [52]:
main_df.head(2)

,equipment_id,maintenance_type,start_time,end_time,maintenance_description,parts_replaced,cost,technician_id,downtime_hours,notes,created_at
0,INJ-002,PREVENTIVE,2024-01-03 10:06:38,2024-01-03 11:38:28,월간 정기점검,NaN,1597990.91,OP015,1.53,다음 정비 시 추가 점검 요망,2026-01-30 02:23:22.393125
1,INJ-002,PREVENTIVE,2024-01-05 15:47:52,2024-01-05 17:47:18,정기 윤활유 교환,"금형, 압력센서",2488706.64,OP011,1.99,NaN,2026-01-30 02:23:22.396196


In [53]:
production_summary=production_df.groupby('equipment_id').agg({'production_id':'count',
                                          'actual_quantity':'sum'})

In [54]:
main_df.head(2)

,equipment_id,maintenance_type,start_time,end_time,maintenance_description,parts_replaced,cost,technician_id,downtime_hours,notes,created_at
0,INJ-002,PREVENTIVE,2024-01-03 10:06:38,2024-01-03 11:38:28,월간 정기점검,NaN,1597990.91,OP015,1.53,다음 정비 시 추가 점검 요망,2026-01-30 02:23:22.393125
1,INJ-002,PREVENTIVE,2024-01-05 15:47:52,2024-01-05 17:47:18,정기 윤활유 교환,"금형, 압력센서",2488706.64,OP011,1.99,NaN,2026-01-30 02:23:22.396196


In [55]:
main_summry =main_df.groupby('equipment_id').agg({'maintenance_type':'count',
                                     'cost':'sum',
                                     'downtime_hours':'sum'})

In [56]:
production_summary.columns = ['생산건수','총생산량']

In [57]:
main_summry.columns = ['정비건수','총정비비용','총정비시간']

In [59]:
eq_df2=eq_df[['equipment_id','equipment_name','equipment_type']]

In [67]:
production_summary

,생산건수,총생산량
equipment_id,,
ASM-001,234,22485
INJ-001,262,28163
INJ-002,430,51958
PRESS-001,468,52069
PRESS-002,478,51929


In [71]:
eq_analysis=pd.merge(eq_df2,production_summary, on = 'equipment_id')

In [73]:
eq_analysis = pd.merge(eq_analysis,main_summry,on = 'equipment_id')

In [74]:
eq_analysis['생산당_정비비용']=eq_analysis['총정비비용']/eq_analysis['총생산량']

In [75]:
import numpy   as np

In [76]:
#분모가 0이 되는 경우도 있기 때문에 
#분모가 0인 경우에는 판다스는 inf or -inf 로 표기 된다
#따라서 이값을 셋팅하는게 현업에서 사용하는 방식
(eq_analysis['총정비비용']/eq_analysis['총생산량']).replace([np.inf,-np.inf],0)

0    1608.200790
1    1444.946659
2     855.869527
3    1365.737480
4    2671.688860
dtype: float64

In [77]:
eq_analysis

,equipment_id,equipment_name,equipment_type,생산건수,총생산량,정비건수,총정비비용,총정비시간,생산당_정비비용
0,INJ-001,사출기 1호기,사출기,262,28163,17,45291758.85,27.85,1608.200790
1,INJ-002,사출기 2호기,사출기,430,51958,23,75076538.50,40.36,1444.946659
2,PRESS-001,프레스 1호기,프레스,468,52069,18,44564270.41,29.69,855.869527
3,PRESS-002,프레스 2호기,프레스,478,51929,21,70921381.60,37.71,1365.737480
4,ASM-001,조립라인 1호기,조립라인,234,22485,19,60072924.01,30.79,2671.688860


In [78]:
#ROI 문제 설비 찾기

In [79]:
#총생산량의 중앙 값보다 작은 것들을 찾고, 총 정비비용의 중앙값보다 작은 것

In [94]:
eq_analysis[(eq_analysis['총생산량']<eq_analysis['총생산량'].median()) & (eq_analysis['총정비비용']>eq_analysis['총정비비용'].median()) ]

,equipment_id,equipment_name,equipment_type,생산건수,총생산량,정비건수,총정비비용,총정비시간,생산당_정비비용
